# Checkpoint 14: check the saved model

Notebook 13 evaluated the selected logistic model on 309 held-out examples with 25 incidents. AP was about 0.9813 and Brier about 0.00376. At the illustrative 20% threshold, it detected 24 incidents with no false positives. Those results describe this synthetic dataset.

Here we load that saved model in a separate Python process and check that predictions remain consistent. We also check duplicates, unavailable corrections and a damaged manifest. This notebook intentionally uses notebook 13's saved artifact; run notebook 13 first if it is missing.

These checks cover the notebook model bundle. The stateful engine was added afterward and has separate tests for memory limits, snapshots and concurrent reload.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dispatch_risk/contracts.py').is_file())
sys.path.insert(0, str(ROOT / 'personal' / 'notebooks' / 'support'))
import workflow as wf
import pandas as pd
import numpy as np
from IPython.display import display
import json,sys,subprocess,tempfile,importlib.util,shutil
artifact = ROOT / 'personal' / 'outputs' / 'learning_model'
if not (artifact / "manifest.json").is_file():
    raise FileNotFoundError("Run notebook 13 to create the model artifact first.")
report = json.loads((ROOT / 'personal' / 'outputs' / 'notebook_results' / "final_evaluation.json").read_text())
print("Using selection:",report["selected_model"],"held-out rows:",report["test"]["rows"])
spec = importlib.util.spec_from_file_location("artifact_verifier",ROOT / 'personal' / 'tools' / 'verify_notebook_artifact.py')
verifier = importlib.util.module_from_spec(spec)
spec.loader.exec_module(verifier)
events = wf.load_jsonl(ROOT / "data" / "events.jsonl")
checkpoints = wf.load_jsonl(ROOT / "data" / "decision_times.jsonl")
# Distributed checkpoints; only metadata determines the selection, not errors/outcomes.
chosen = checkpoints[::max(1,len(checkpoints)//12)][:12]
ids = {d["shipment_id"] for d in chosen}
probe = {"events":[e for e in events if e["shipment_id"] in ids],"checkpoints":chosen}
probe_path = ROOT / 'personal' / 'outputs' / 'notebook_results' / "artifact_probe.json"
wf.save_json(probe_path,probe)
parent = verifier.score(artifact,probe)
expected = json.dumps(parent,sort_keys=True,separators=(",",":"),allow_nan=False)+"\n"
with tempfile.TemporaryDirectory(prefix="risk-fresh-process-") as unrelated:
    command = [sys.executable,str(ROOT / 'personal' / 'tools' / 'verify_notebook_artifact.py'),"--artifact",str(artifact),"--probe",str(probe_path)]
    first = subprocess.run(command,cwd=unrelated,text=True,capture_output=True,check=True)
    second = subprocess.run(command,cwd=unrelated,text=True,capture_output=True,check=True)
    assert first.stdout == second.stdout == expected
print("Fresh processes exactly matched parent-process probability and feature-digest JSON for",len(chosen),"checkpoints.")


Using selection: logistic_regression held-out rows: 309
Fresh processes exactly matched parent-process probability and feature-digest JSON for 12 checkpoints.


## 1. Revision and duplication checks through the saved model

These tests replay feature reconstruction from raw records. They do not test online state eviction or snapshot restore. Historical feature extraction must ignore deliveries unavailable at the checkpoint and collapse duplicates exactly as it did during training.


In [2]:
repeated = {"events":probe["events"]*2,"checkpoints":probe["checkpoints"]}
assert verifier.score(artifact,repeated) == parent
reversed_probe = {"events":list(reversed(probe["events"])),"checkpoints":probe["checkpoints"]}
assert verifier.score(artifact,reversed_probe) == parent
latest_checkpoint = max(wf.utc(d["decision_time"]) for d in chosen)
original = probe["events"][0]
higher = max(e["revision"] for e in probe["events"] if e["event_id"] == original["event_id"])+1
future_correction = {**original,"revision":higher,"value":999.,"received_at":(latest_checkpoint+wf.timedelta(days=2)).isoformat()}
assert verifier.score(artifact,{"events":probe["events"]+[future_correction],"checkpoints":chosen}) == parent
with tempfile.TemporaryDirectory(prefix="risk-damaged-artifact-") as destination:
    broken = Path(destination)
    for name in ["model.joblib","feature_runtime.py","manifest.json"]:
        shutil.copyfile(artifact/name,broken/name)
    metadata=json.loads((broken/"manifest.json").read_text())
    metadata["checksums"]["model.joblib"]="0"*64
    wf.save_json(broken/"manifest.json",metadata)
    try:
        verifier.score(broken,probe)
    except ValueError as error:
        assert "checksum" in str(error)
    else:
        raise AssertionError("Damaged manifest was accepted")
print("Passed: duplicate deliveries, historical file permutation, future correction, damaged-artifact rejection.")


Passed: duplicate deliveries, historical file permutation, future correction, damaged-artifact rejection.


## 2. Smoke-test unfamiliar generated inputs

Generate a different small stream into a temporary folder, then score all its checkpoints with the existing saved model. Do not retrain, select models, or optimize on this stream. We only verify that feature and artifact code works beyond original IDs and emits finite probabilities.


In [3]:
with tempfile.TemporaryDirectory(prefix="risk-new-stream-") as directory:
    subprocess.run([sys.executable,str(ROOT/"tools/generate_dataset.py"),"--seed","1730","--shipments","80","--output",directory],capture_output=True,text=True,check=True)
    fresh = {"events":wf.load_jsonl(Path(directory)/"events.jsonl"),"checkpoints":wf.load_jsonl(Path(directory)/"decision_times.jsonl")}
    answer = verifier.score(artifact,fresh)
    assert len(answer["probabilities"]) == len(fresh["checkpoints"])
    print("Different generated stream:",len(fresh["events"]),"deliveries;",len(answer["probabilities"]),"valid checkpoint probabilities.")
checks = {"fresh_process_exact_match":True,"duplicate_invariance":True,"historical_permutation_invariance":True,"future_correction_invariance":True,"damaged_artifact_rejected":True,"new_generated_stream_scored":True,"public_risk_engine_tested":False}
wf.save_json(ROOT/"personal/outputs/notebook_results/artifact_checks.json",checks)


Different generated stream: 1467 deliveries; 240 valid checkpoint probabilities.


## 3. How the experiment became the Python implementation

The model checks established that a separate process could reconstruct features and load the saved model. The next step was to implement the stateful behavior required by the assignment.

| Requirement | Implemented in | What to inspect |
|---|---|---|
| Training rows and labels | `training.py` | Available reports, mature outcomes and excluded-row counts |
| Model fitting and export | `training.py`, `model.py` | Chronological evaluation and fitted preprocessing |
| Ingest and score | `engine.py` | Delivery identity, retained revisions and explicit scoring time |
| Memory limits | `engine.py` | Shipment eviction, record cap and incomplete-history reasons |
| Snapshot and restore | `engine.py` | Consistent state capture and deterministic serialization |
| Concurrent reload | `engine.py` | Validate first, swap model, retain telemetry on success or failure |
| Verification | `tests/test_engine.py`, `tests/test_training.py` | Replay, corruption, concurrency and training checks |

The public interface is now implemented. See [DECISIONS.md](../DECISIONS.md) for its policies and [RUNBOOK.md](../RUNBOOK.md) for commands. The notebook checks above still cover only the model bundle; they do not replace the engine tests.

Two limits matter when explaining the handoff. The builder has no explicit observation-cutoff argument, so its metadata records the chosen assumption. The engine has finite memory, so it flags predictions affected by discarded history.

**Interview practice:** explain the two clocks, step through a delayed correction, describe the split and assumed negatives, and explain why logistic regression met the selection rule. Then use the engine tests to demonstrate the operational guarantees.

**Try explaining this:** what does loading the model in a fresh process prove, and what still needs a stateful engine test?